# ES 678: setup, EDA and pilot runs

Runtime: **GPU** (Runtime > Change runtime type). Sections:

- **A** every session: mount Drive, get the code, stage the image cache on local disk
- **B** once: download Waterbirds, build the splits and the cache
- **C** EDA and integrity checks
- **D** timing pilot (300 steps) with a projection of the full compute budget
- **E** ERM pilot at rho = 0.95 (validation curves only, never test)
- **F** official-split anchor, the one sanctioned early look at test numbers

If you are not using GitHub yet, upload the repository zip to Colab and unzip it
to `/content/shortcut-reliability` instead of cloning.

## A. Every session

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE = "/content/drive/MyDrive/es678"     # persistent: cache, runs, reports
REPO_URL = "https://github.com/YOUR_USERNAME/shortcut-reliability.git"   # edit this
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
%cd /content
if not os.path.exists("/content/shortcut-reliability"):
    !git clone -q $REPO_URL shortcut-reliability
%cd /content/shortcut-reliability
!git pull -q
!git log -1 --oneline
!pip install -q -e .

In [ ]:
SPLITS = "splits/controlled_v1.json"
DATA = "/content/data/waterbird_complete95_forest2water2"
CACHE_DRIVE = f"{DRIVE}/cache_256"     # built once (section B), kept on Drive
CACHE = "/content/cache_256"           # local copy: much faster reads than Drive
RUNS = f"{DRIVE}/runs"
REPORTS = f"{DRIVE}/reports"
if os.path.exists(f"{CACHE_DRIVE}/manifest.json") and not os.path.exists(f"{CACHE}/manifest.json"):
    !mkdir -p $CACHE && cp -r $CACHE_DRIVE/. $CACHE/
print("local cache ready:", os.path.exists(f"{CACHE}/manifest.json"))

## B. One-time data preparation

`make_splits.py` fails loudly if the per-group counts differ from the official
release. After this section, commit `splits/controlled_v1.json` to GitHub and
copy its sha256 into `PREREGISTRATION.md`. Never regenerate it afterwards.

In [ ]:
!bash scripts/download_waterbirds.sh /content/data

In [ ]:
if os.path.exists(SPLITS):
    print(f"{SPLITS} exists. It is part of the experimental record: do not regenerate it.")
else:
    !python scripts/make_splits.py --data_root $DATA --out $SPLITS
    !cp $SPLITS $DRIVE/

In [ ]:
# Build on local disk (memory-mapped writes to Drive are slow), then keep a copy on Drive.
!python scripts/build_cache.py --data_root $DATA --out_dir $CACHE
!mkdir -p $CACHE_DRIVE && cp -r $CACHE/. $CACHE_DRIVE/

## C. EDA and integrity checks

In [ ]:
!python scripts/eda.py --cache_dir $CACHE --splits $SPLITS --out_dir $REPORTS/eda
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{REPORTS}/eda/eda_report.md").read().split("## 6.")[0]))
display(Image(f"{REPORTS}/eda/group_samples.png"))
display(Image(f"{REPORTS}/eda/image_sizes.png"))

## D. Timing pilot

300 steps of ERM, then a projection for a full 2,000-step run and for the main
grid (5 rho levels x 4 trained methods x seeds). These numbers decide between
ResNet-50 and ResNet-18 and between 3 and 5 seeds.

In [ ]:
!python scripts/train.py --cache_dir $CACHE --splits_path $SPLITS --rho 0.95 --data_seed 0 \
    --steps 300 --eval_every 100 --dense_eval_every 0 --tag timing --out_dir $RUNS

In [ ]:
import glob, json
from srl.train import make_eval_steps
run = sorted(glob.glob(f"{RUNS}/*_timing"))[-1]
s = json.load(open(f"{run}/summary.json"))
sec_per_step = s["train_seconds"] / 300
sec_per_eval = s["eval_seconds"] / s["n_evals"]
n_evals_full = len(make_eval_steps(2000, 100, 25, 200))
per_run = 2000 * sec_per_step + n_evals_full * sec_per_eval
print(f"training {s['train_images_per_second']:.0f} img/s; {sec_per_eval:.1f} s per evaluation; "
      f"peak GPU memory {s.get('max_gpu_memory_gb')} GB")
print(f"projected full run (2000 steps, {n_evals_full} evaluations): {per_run / 60:.1f} min")
for seeds in (3, 5):
    n_runs = 5 * 4 * seeds
    print(f"main grid with {seeds} seeds ({n_runs} runs): {n_runs * per_run / 3600:.1f} GPU-hours")

## E. ERM pilot at rho = 0.95

Look at validation curves only. Check that (1) training accuracy reaches 1,
(2) val_bal worst-group accuracy sits well below val_id accuracy (the shortcut),
and (3) the curves have flattened by step 2,000. If they have not, raise
`--steps` for every run before you preregister.

In [ ]:
!python scripts/train.py --cache_dir $CACHE --splits_path $SPLITS --rho 0.95 --data_seed 0 --out_dir $RUNS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
run = sorted(r for r in glob.glob(f"{RUNS}/controlled_rho0.95_ds0_ts0_erm_*") if not r.endswith("_timing"))[-1]
log = pd.read_csv(f"{run}/train_log.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for col in ["train_acc", "train_wga", "val_bal_acc", "val_bal_wga", "val_id_acc", "val_id_worst_class_acc"]:
    ax[0].plot(log.step, log[col], marker=".", label=col)
ax[0].set_xlabel("step"); ax[0].set_ylim(0, 1.02); ax[0].legend(fontsize=8)
ax[0].set_title("training and validation metrics (no test)")
ax[1].plot(log.step, log.train_loss, marker="."); ax[1].set_yscale("log")
ax[1].set_xlabel("step"); ax[1].set_title("training loss")
plt.tight_layout(); plt.show()
!python scripts/final_eval.py --runs "$run"

## F. Official-split anchor

Three ERM seeds on the official split. Unsealing test results here is allowed
because it is a pipeline check against published numbers, not a decision about
the controlled study. Do not tune anything on these test numbers.

In [ ]:
for seed in (0, 1, 2):
    !python scripts/train.py --cache_dir $CACHE --split official --data_seed $seed --out_dir $RUNS

In [ ]:
!python scripts/final_eval.py --runs "$RUNS/official_*" --protocols S1 S2 --unseal --out_csv $REPORTS/anchor_official.csv